
# ========================================
# DATA EXPORT FOR DASHBOARD
# Run this BEFORE opening the dashboard
# ========================================

In [1]:
# CELL 1: Setup
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import json
import os
from datetime import datetime

print("✅ Setup complete\n")

Mounted at /content/drive
✅ Setup complete



In [2]:
# CELL 2: Load Enriched Data
EXPORT_DIR = '/content/drive/MyDrive/vendor-analysis-project/data/exports'

enriched_file = f'{EXPORT_DIR}/vendors_enriched.csv'

print(f"Loading data from: {enriched_file}\n")
df = pd.read_csv(enriched_file, low_memory=False)


print(f"✅ Loaded {len(df):,} vendors\n")



Loading data from: /content/drive/MyDrive/vendor-analysis-project/data/exports/vendors_enriched.csv

✅ Loaded 100 vendors



In [3]:
# CELL 3: Prepare All Dashboard Data
print("Preparing dashboard data...\n")

# 1. Main metrics
main_metrics = {
    'totalVendors': int(len(df)),
    'totalSpend': float(df['total_spend'].sum()),
    'avgSpend': float(df['total_spend'].mean()),
    'maxSpend': float(df['total_spend'].max()),
    'yoyGrowth': float((df['spend_2024'].sum() - df['spend_2023'].sum())/df['spend_2023'].sum()*100) if df['spend_2023'].sum() > 0 else 0,
    'healthyVendors': int(len(df[df['vendor_health_status']=='Healthy'])),
    'atRiskVendors': int(len(df[df['vendor_health_status']=='At Risk'])),
    'criticalVendors': int(len(df[df['vendor_health_status']=='Critical'])),
    'statesCovered': int(df[df['state']!='']['state'].nunique()),
    'withEmail': int(len(df[df['primary_email']!=''])),
    'withPhone': int(len(df[df['phone']!=''])),
    'avgPerformance': float(df['performance_score'].mean()),
    'avgActivity': float(df['activity_score'].mean()),
    'spend2024': float(df['spend_2024'].sum()),
    'spend2023': float(df['spend_2023'].sum()),
    'spend2022': float(df['spend_2022'].sum()),
}

print("✓ Main metrics calculated")

# 2. Tier Distribution
tier_dist = []
for tier in df['vendor_tier'].unique():
    tier_df = df[df['vendor_tier'] == tier]
    tier_dist.append({
        'name': tier,
        'value': int(tier_df.shape[0]),
        'spend': float(tier_df['total_spend'].sum())
    })

tier_data = sorted(tier_dist, key=lambda x: x['spend'], reverse=True)
print("✓ Tier distribution calculated")

# 3. Health Status Data
health_data = [
    {'name': 'Healthy', 'value': int(len(df[df['vendor_health_status']=='Healthy'])), 'fill': '#2ecc71'},
    {'name': 'At Risk', 'value': int(len(df[df['vendor_health_status']=='At Risk'])), 'fill': '#f39c12'},
    {'name': 'Critical', 'value': int(len(df[df['vendor_health_status']=='Critical'])), 'fill': '#e74c3c'},
    {'name': 'Inactive', 'value': int(len(df[df['vendor_health_status']=='Inactive'])), 'fill': '#95a5a6'}
]

print("✓ Health status data calculated")

# 4. Spend Trend (2022-2024)
spend_trend_data = [
    {'year': '2022', 'spend': float(df['spend_2022'].sum())},
    {'year': '2023', 'spend': float(df['spend_2023'].sum())},
    {'year': '2024', 'spend': float(df['spend_2024'].sum())}
]

print("✓ Spend trend calculated")

# 5. Top 15 Vendors
top_vendors = df.nlargest(15, 'total_spend')[['vendor_name', 'total_spend', 'state', 'spend_growth_2023_2024']].copy()
top_vendors_list = []
for idx, row in top_vendors.iterrows():
    top_vendors_list.append({
        'name': str(row['vendor_name']),
        'spend': float(row['total_spend']),
        'growth': float(row['spend_growth_2023_2024']),
        'state': str(row['state'])
    })

print("✓ Top vendors calculated")

# 6. Top 15 Growing Vendors
growing_vendors = df[df['spend_2023'] > 0].nlargest(15, 'spend_growth_2023_2024')[['vendor_name', 'spend_2023', 'spend_2024', 'spend_growth_2023_2024']].copy()
growing_vendors_list = []
for idx, row in growing_vendors.iterrows():
    growing_vendors_list.append({
        'name': str(row['vendor_name']),
        'spend2023': float(row['spend_2023']),
        'spend2024': float(row['spend_2024']),
        'growth': float(row['spend_growth_2023_2024'])
    })

print("✓ Growing vendors calculated")

# 7. Top 10 Declining Vendors
declining_vendors = df[df['spend_2023'] > 0].nsmallest(10, 'spend_growth_2023_2024')[['vendor_name', 'spend_2023', 'spend_2024', 'spend_growth_2023_2024']].copy()
declining_vendors_list = []
for idx, row in declining_vendors.iterrows():
    declining_vendors_list.append({
        'name': str(row['vendor_name']),
        'spend2023': float(row['spend_2023']),
        'spend2024': float(row['spend_2024']),
        'growth': float(row['spend_growth_2023_2024'])
    })

print("✓ Declining vendors calculated")

# 8. State Analysis (Top 15)
state_analysis = df[df['state']!=''].groupby('state').agg({
    'vendor_name': 'count',
    'total_spend': 'sum'
}).reset_index()
state_analysis.columns = ['state', 'vendors', 'spend']
state_analysis = state_analysis.nlargest(15, 'spend')
state_data = []
for idx, row in state_analysis.iterrows():
    state_data.append({
        'name': str(row['state']),
        'vendors': int(row['vendors']),
        'spend': float(row['spend'])
    })

print("✓ State analysis calculated")

# 9. Vendor Type Analysis
vendor_type_analysis = df[df['vendor_type']!=''].groupby('vendor_type').agg({
    'vendor_name': 'count',
    'total_spend': 'sum'
}).reset_index()
vendor_type_analysis.columns = ['type', 'count', 'spend']
vendor_type_analysis = vendor_type_analysis.nlargest(10, 'spend')
vendor_type_data = []
for idx, row in vendor_type_analysis.iterrows():
    vendor_type_data.append({
        'name': str(row['type']),
        'vendors': int(row['count']),
        'spend': float(row['spend'])
    })

print("✓ Vendor type analysis calculated")

# 10. Risk Category Analysis
risk_analysis = df['risk_category'].value_counts().to_dict()
risk_data = []
for category, count in risk_analysis.items():
    risk_data.append({
        'name': str(category),
        'value': int(count)
    })

print("✓ Risk category analysis calculated")

# 11. Engagement Distribution
engagement_analysis = df['engagement_level'].value_counts().to_dict()
engagement_data = []
for level, count in engagement_analysis.items():
    engagement_data.append({
        'name': str(level),
        'value': int(count)
    })

print("✓ Engagement level analysis calculated")

# 12. Performance Score Distribution
performance_bins = [0, 25, 50, 75, 100]
performance_dist = pd.cut(df['performance_score'], bins=performance_bins).value_counts().sort_index()
performance_data = []
for idx, count in performance_dist.items():
    performance_data.append({
        'range': f"{int(idx.left)}-{int(idx.right)}",
        'count': int(count)
    })

print("✓ Performance score distribution calculated")


Preparing dashboard data...

✓ Main metrics calculated
✓ Tier distribution calculated
✓ Health status data calculated
✓ Spend trend calculated
✓ Top vendors calculated
✓ Growing vendors calculated
✓ Declining vendors calculated
✓ State analysis calculated
✓ Vendor type analysis calculated
✓ Risk category analysis calculated
✓ Engagement level analysis calculated
✓ Performance score distribution calculated


In [4]:
# CELL 4: Combine All Data
all_dashboard_data = {
    'exportDate': datetime.now().isoformat(),
    'mainMetrics': main_metrics,
    'tierData': tier_data,
    'healthData': health_data,
    'spendTrendData': spend_trend_data,
    'topVendors': top_vendors_list,
    'growingVendors': growing_vendors_list,
    'decliningVendors': declining_vendors_list,
    'stateData': state_data,
    'vendorTypeData': vendor_type_data,
    'riskData': risk_data,
    'engagementData': engagement_data,
    'performanceData': performance_data,
}

print("\n✅ All data prepared!\n")



✅ All data prepared!



In [5]:
# CELL 5: Save as JSON
json_file = f'{EXPORT_DIR}/dashboard_data.json'
with open(json_file, 'w') as f:
    json.dump(all_dashboard_data, f, indent=2)

print(f"✅ Dashboard data saved to: {json_file}")


✅ Dashboard data saved to: /content/drive/MyDrive/vendor-analysis-project/data/exports/dashboard_data.json


In [6]:
# CELL 6: Create Summary
print("\n" + "="*80)
print("DASHBOARD DATA SUMMARY")
print("="*80)
print(f"\nMain Metrics:")
print(f"  Total Vendors: {main_metrics['totalVendors']:,}")
print(f"  Total Spend: ${main_metrics['totalSpend']:,.0f}")
print(f"  Average Spend: ${main_metrics['avgSpend']:,.0f}")
print(f"  YoY Growth: {main_metrics['yoyGrowth']:.2f}%")
print(f"\nData Collections:")
print(f"  Tier Distribution: {len(tier_data)} tiers")
print(f"  Top Vendors: {len(top_vendors_list)} vendors")
print(f"  Growing Vendors: {len(growing_vendors_list)} vendors")
print(f"  Declining Vendors: {len(declining_vendors_list)} vendors")
print(f"  States: {len(state_data)} states")
print(f"  Vendor Types: {len(vendor_type_data)} types")

print("\n✅ Ready to use in dashboard!")
print(f"\nFile location: {json_file}")


DASHBOARD DATA SUMMARY

Main Metrics:
  Total Vendors: 100
  Total Spend: $66,043,915
  Average Spend: $660,439
  YoY Growth: 83.53%

Data Collections:
  Tier Distribution: 5 tiers
  Top Vendors: 15 vendors
  Growing Vendors: 15 vendors
  Declining Vendors: 10 vendors
  States: 15 states
  Vendor Types: 6 types

✅ Ready to use in dashboard!

File location: /content/drive/MyDrive/vendor-analysis-project/data/exports/dashboard_data.json
